In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
from statsmodels.stats.stattools import durbin_watson
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [8]:
df = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/preprocessed_not_standardized.csv')

In [9]:
df = df[['period','appreciation_return', 'available_df_direct', 'demand_sf', 'net_delivered_sf', 'bond_yield_10yr_x', 'three_yr_inf_exp', 'fed_funds_rate', 'cmbs_to_gdp', 'market_cap_rate']]

In [10]:
target_variable = 'market_cap_rate'

In [11]:
#use the period column as index
df['period'] = pd.to_datetime(df['period'])
df.set_index('period', inplace=True)

In [18]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

# --- Part 1: Transformation and Stationarity Checks ---

def adf_test(series, name=''):
    """Perform ADF test and return results as a dictionary."""
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'Variable': name,
        'ADF Statistic': result[0],
        'p-value': result[1],
        'Stationary': result[1] <= 0.05
    }

def transform_and_check_stationarity(df, target_variable, cv_threshold=0.5, seasonal_period=8):
    """Transform data and check stationarity before and after each step."""
    # Initial Stationarity Check
    print("\n--- INITIAL STATIONARITY CHECK ---")
    initial_results = [adf_test(df[col], col) for col in df.select_dtypes(include=np.number).columns]
    print(pd.DataFrame(initial_results))
    
    # Create a new DataFrame for transformed columns
    df_transformed = pd.DataFrame(index=df.index)
    target_transformation = None
    
    # A. Log Transformation (if CV > threshold)
    print("\n--- LOG TRANSFORMATION (IF NEEDED) ---")
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    for col in numeric_cols:
        series = df[col].dropna()
        if series.min() > 0:  # Log only if all values are positive
            cv = series.std() / series.mean()
            if cv > cv_threshold:
                df_transformed[f'log_{col}'] = np.log(series)
                if col == target_variable:
                    target_transformation = f'log_{col}'
            else:
                df_transformed[f'orig_{col}'] = series
                if col == target_variable:
                    target_transformation = f'orig_{col}'
        else:
            df_transformed[f'orig_{col}'] = series
            if col == target_variable:
                target_transformation = f'orig_{col}'

    # Check Stationarity After Log Transformation
    print("\n--- STATIONARITY CHECK AFTER LOG TRANSFORMATION ---")
    log_results = [adf_test(df_transformed[col], col) for col in df_transformed.columns]
    print(pd.DataFrame(log_results))
    
    # B. First Differencing (If Non-Stationary)
    print("\n--- FIRST DIFFERENCING (IF NEEDED) ---")
    df_diff = df_transformed.copy()
    for col in df_transformed.columns:
        series = df_transformed[col].dropna()
        adf_pvalue = adfuller(series)[1]
        if adf_pvalue > 0.05:
            base_name = col.split('_', 1)[1] if '_' in col else col
            df_diff[f'd1_{base_name}'] = series.diff()
            if base_name == target_variable:
                target_transformation = f'd1_{base_name}'
        else:
            df_diff[col] = series

    # Drop NA values created by differencing
    df_diff = df_diff.dropna()

    # Check Stationarity After First Differencing
    print("\n--- STATIONARITY CHECK AFTER FIRST DIFFERENCING ---")
    diff_results = [adf_test(df_diff[col], col) for col in df_diff.columns]
    print(pd.DataFrame(diff_results))

    # C. Seasonal Differencing (If Needed)
    print("\n--- SEASONAL DIFFERENCING (IF NEEDED) ---")
    df_seasonal = df_diff.copy()
    for col in df_diff.columns:
        series = df_diff[col].dropna()
        adf_pvalue = adfuller(series)[1]
        if adf_pvalue > 0.05:
            df_seasonal[f'sd{seasonal_period}_{col}'] = series.diff(seasonal_period)
            if col == target_variable:
                target_transformation = f'sd{seasonal_period}_{col}'
                
    # Drop any NAs from seasonal differencing
    df_seasonal = df_seasonal.dropna()

    # Final Stationarity Check
    print("\n--- FINAL STATIONARITY CHECK AFTER SEASONAL DIFFERENCING ---")
    final_results = [adf_test(df_seasonal[col], col) for col in df_seasonal.columns]
    print(pd.DataFrame(final_results))

    print(f"\n✅ FINAL TARGET TRANSFORMATION: {target_transformation}")

    return df_seasonal, target_transformation

# --- Part 2: VAR Modeling ---

def run_var_model(df_stationary):
    """
    Fits a VAR model on the stationary DataFrame.
    """
    # Ensure data is sorted by time (assuming the index is a datetime index)
    df_stationary = df_stationary.sort_index()
    
    # Initialize and select the optimal lag order using VAR
    model = VAR(df_stationary)
    lag_order_results = model.select_order(maxlags=4)
    print("\n--- Optimal Lag Order Selection ---")
    print(lag_order_results.summary())
    
    # Choose lag order based on AIC (you can choose another criterion if needed)
    chosen_lag = lag_order_results.selected_orders['aic']
    print(f"\nChosen lag order based on AIC: {chosen_lag}")
    
    # Fit the VAR model with the chosen lag order
    var_model = model.fit(chosen_lag)
    
    print("\n--- VAR Model Summary ---")
    print(var_model.summary())
    
    return var_model

# --- Example Usage ---

# Assume 'df' is your original DataFrame with a time index and columns (including 'market_cap_rate')
# For example:
# df = pd.read_csv('your_data.csv', index_col='date', parse_dates=True)

# Step 1: Transform data and ensure stationarity
df_stationary, target_transformation = transform_and_check_stationarity(df, target_variable='market_cap_rate', cv_threshold=0.5, seasonal_period=8)

# Step 2: Run the VAR model on the transformed, stationary data
var_model = run_var_model(df_stationary)



--- INITIAL STATIONARITY CHECK ---


              Variable  ADF Statistic   p-value  Stationary
0  appreciation_return      -2.653460  0.082444       False
1  available_df_direct      -2.252702  0.187676       False
2            demand_sf      -1.888699  0.337396       False
3     net_delivered_sf      -2.573779  0.098539       False
4    bond_yield_10yr_x      -2.360832  0.153098       False
5     three_yr_inf_exp      -2.693158  0.075230       False
6       fed_funds_rate      -4.129793  0.000864        True
7          cmbs_to_gdp      -2.156282  0.222493       False
8      market_cap_rate      -2.961104  0.038690        True

--- LOG TRANSFORMATION (IF NEEDED) ---

--- STATIONARITY CHECK AFTER LOG TRANSFORMATION ---
                   Variable  ADF Statistic   p-value  Stationary
0  orig_appreciation_return      -2.653460  0.082444       False
1  orig_available_df_direct      -2.252702  0.187676       False
2            orig_demand_sf      -1.888699  0.337396       False
3     orig_net_delivered_sf      -2.573779  0.0

/Users/elyas/Applications/anaconda3/envs/gatech_02/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)


ValueError: maxlags is too large for the number of observations and the number of equations. The largest model cannot be estimated.

In [16]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

# --- Step 0: Read and Preprocess the Data ---
df = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/preprocessed_not_standardized.csv')

# Subset the desired columns
df = df[['period',
         'appreciation_return',
         'available_df_direct',
         'demand_sf',
         'net_delivered_sf',
         'bond_yield_10yr_x',
         'three_yr_inf_exp',
         'fed_funds_rate',
         'cmbs_to_gdp',
         'market_cap_rate']]

# Convert the 'period' column to datetime and set it as index
df['period'] = pd.to_datetime(df['period'])
df.set_index('period', inplace=True)

# --- Part 1: Transformation and Stationarity Checks ---

def adf_test(series, name=''):
    """Perform ADF test and return results as a dictionary."""
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'Variable': name,
        'ADF Statistic': result[0],
        'p-value': result[1],
        'Stationary': result[1] <= 0.05
    }

def transform_and_check_stationarity(df, target_variable, cv_threshold=0.5, seasonal_period=8):
    """Transform data and check stationarity before and after each step."""
    # Initial Stationarity Check
    print("\n--- INITIAL STATIONARITY CHECK ---")
    initial_results = [adf_test(df[col], col) for col in df.select_dtypes(include=np.number).columns]
    print(pd.DataFrame(initial_results))
    
    # Create a new DataFrame for transformed columns
    df_transformed = pd.DataFrame(index=df.index)
    target_transformation = None
    
    # A. Log Transformation (if CV > threshold)
    print("\n--- LOG TRANSFORMATION (IF NEEDED) ---")
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    for col in numeric_cols:
        series = df[col].dropna()
        if series.min() > 0:  # Only apply log if all values are positive
            cv = series.std() / series.mean()
            if cv > cv_threshold:
                df_transformed[f'log_{col}'] = np.log(series)
                if col == target_variable:
                    target_transformation = f'log_{col}'
            else:
                df_transformed[f'orig_{col}'] = series
                if col == target_variable:
                    target_transformation = f'orig_{col}'
        else:
            df_transformed[f'orig_{col}'] = series
            if col == target_variable:
                target_transformation = f'orig_{col}'

    print("\n--- STATIONARITY CHECK AFTER LOG TRANSFORMATION ---")
    log_results = [adf_test(df_transformed[col], col) for col in df_transformed.columns]
    print(pd.DataFrame(log_results))
    
    # B. First Differencing (If Non-Stationary)
    print("\n--- FIRST DIFFERENCING (IF NEEDED) ---")
    df_diff = df_transformed.copy()
    for col in df_transformed.columns:
        series = df_transformed[col].dropna()
        adf_pvalue = adfuller(series)[1]
        if adf_pvalue > 0.05:
            base_name = col.split('_', 1)[1] if '_' in col else col
            df_diff[f'd1_{base_name}'] = series.diff()
 


In [20]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

# --- Step 0: Read and Preprocess the Data ---
df = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/preprocessed_not_standardized.csv')

# Subset the desired columns
df = df[['period',
         'appreciation_return',
         'available_df_direct',
         'demand_sf',
         'net_delivered_sf',
         'bond_yield_10yr_x',
         'three_yr_inf_exp',
         'fed_funds_rate',
         'cmbs_to_gdp',
         'market_cap_rate']]

# Convert the 'period' column to datetime and set it as index
df['period'] = pd.to_datetime(df['period'])
df.set_index('period', inplace=True)

# --- Part 1: Transformation and Stationarity Checks ---

def adf_test(series, name=''):
    """Perform ADF test and return results as a dictionary."""
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'Variable': name,
        'ADF Statistic': result[0],
        'p-value': result[1],
        'Stationary': result[1] <= 0.05
    }

def transform_and_check_stationarity(df, target_variable, cv_threshold=0.5, seasonal_period=8):
    """Transform data and check stationarity before and after each step."""
    # Initial Stationarity Check
    print("\n--- INITIAL STATIONARITY CHECK ---")
    initial_results = [adf_test(df[col], col) for col in df.select_dtypes(include=np.number).columns]
    print(pd.DataFrame(initial_results))
    
    # Create a new DataFrame for transformed columns
    df_transformed = pd.DataFrame(index=df.index)
    target_transformation = None
    
    # A. Log Transformation (if CV > threshold)
    print("\n--- LOG TRANSFORMATION (IF NEEDED) ---")
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    for col in numeric_cols:
        series = df[col].dropna()
        if series.min() > 0:  # Only apply log if all values are positive
            cv = series.std() / series.mean()
            if cv > cv_threshold:
                df_transformed[f'log_{col}'] = np.log(series)
                if col == target_variable:
                    target_transformation = f'log_{col}'
            else:
                df_transformed[f'orig_{col}'] = series
                if col == target_variable:
                    target_transformation = f'orig_{col}'
        else:
            df_transformed[f'orig_{col}'] = series
            if col == target_variable:
                target_transformation = f'orig_{col}'

    print("\n--- STATIONARITY CHECK AFTER LOG TRANSFORMATION ---")
    log_results = [adf_test(df_transformed[col], col) for col in df_transformed.columns]
    print(pd.DataFrame(log_results))
    
    # B. First Differencing (If Non-Stationary)
    print("\n--- FIRST DIFFERENCING (IF NEEDED) ---")
    df_diff = df_transformed.copy()
    for col in df_transformed.columns:
        series = df_transformed[col].dropna()
        adf_pvalue = adfuller(series)[1]
        if adf_pvalue > 0.05:
            base_name = col.split('_', 1)[1] if '_' in col else col
            df_diff[f'd1_{base_name}'] = series.diff()
            if base_name == target_variable:
                target_transformation = f'd1_{base_name}'
        else:
            df_diff[col] = series
    df_diff = df_diff.dropna()

    print("\n--- STATIONARITY CHECK AFTER FIRST DIFFERENCING ---")
    diff_results = [adf_test(df_diff[col], col) for col in df_diff.columns]
    print(pd.DataFrame(diff_results))

    # C. Seasonal Differencing (If Needed)
    print("\n--- SEASONAL DIFFERENCING (IF NEEDED) ---")
    df_seasonal = df_diff.copy()
    for col in df_diff.columns:
        series = df_diff[col].dropna()
        adf_pvalue = adfuller(series)[1]
        if adf_pvalue > 0.05:
            df_seasonal[f'sd{seasonal_period}_{col}'] = series.diff(seasonal_period)
            if col == target_variable:
                target_transformation = f'sd{seasonal_period}_{col}'
    df_seasonal = df_seasonal.dropna()

    print("\n--- FINAL STATIONARITY CHECK AFTER SEASONAL DIFFERENCING ---")
    final_results = [adf_test(df_seasonal[col], col) for col in df_seasonal.columns]
    print(pd.DataFrame(final_results))

    print(f"\n✅ FINAL TARGET TRANSFORMATION: {target_transformation}")

    return df_seasonal, target_transformation

# --- Part 2: VAR Modeling ---

def run_var_model(df_stationary):
    """
    Fits a VAR model on the stationary DataFrame.
    """
    # Ensure data is sorted by time
    df_stationary = df_stationary.sort_index()
    
    # To keep the number of equations manageable, select only columns that represent the final transformation.
    # Here we assume these columns start with 'd1_' or 'sd'. If none, fall back to all columns.
    final_cols = [col for col in df_stationary.columns if col.startswith("d1_") or col.startswith("sd")]
    if not final_cols:
        final_cols = df_stationary.columns.tolist()
    df_var = df_stationary[final_cols]
    
    # Recalculate sample size and number of equations for the VAR model.
    n_obs, n_eq = df_var.shape[0], df_var.shape[1]
    
    # Compute allowed maxlags. Adjust the formula if necessary.
    allowed_maxlags = int(np.floor((n_obs - 1) / (n_eq + 1)))
    if allowed_maxlags < 1:
         raise ValueError("Not enough observations to estimate a VAR model.")
    maxlags_to_use = min(4, allowed_maxlags)
    print(f"\nUsing maxlags={maxlags_to_use} for lag order selection (Allowed maxlags based on data: {allowed_maxlags}).")
    
    # Initialize the VAR model with the selected variables.
    model = VAR(df_var)
    lag_order_results = model.select_order(maxlags=maxlags_to_use)
    print("\n--- Optimal Lag Order Selection ---")
    print(lag_order_results.summary())
    
    chosen_lag = lag_order_results.selected_orders['aic']
    # If AIC does not select a lag, fallback to lag 1.
    if chosen_lag is None:
        chosen_lag = 1
    print(f"\nChosen lag order based on AIC: {chosen_lag}")
    
    # Fit the VAR model with the chosen lag order.
    var_model = model.fit(chosen_lag)
    
    print("\n--- VAR Model Summary ---")
    print(var_model.summary())
    
    return var_model

# --- Execution: End-to-End Process ---

# Step 1: Transform the data and check stationarity.
df_stationary, target_transformation = transform_and_check_stationarity(
    df, 
    target_variable='market_cap_rate', 
    cv_threshold=0.5, 
    seasonal_period=4
)

# Step 2: Run the VAR model on the stationary data.
var_model = run_var_model(df_stationary)



--- INITIAL STATIONARITY CHECK ---


              Variable  ADF Statistic   p-value  Stationary
0  appreciation_return      -2.653460  0.082444       False
1  available_df_direct      -2.252702  0.187676       False
2            demand_sf      -1.888699  0.337396       False
3     net_delivered_sf      -2.573779  0.098539       False
4    bond_yield_10yr_x      -2.360832  0.153098       False
5     three_yr_inf_exp      -2.693158  0.075230       False
6       fed_funds_rate      -4.129793  0.000864        True
7          cmbs_to_gdp      -2.156282  0.222493       False
8      market_cap_rate      -2.961104  0.038690        True

--- LOG TRANSFORMATION (IF NEEDED) ---

--- STATIONARITY CHECK AFTER LOG TRANSFORMATION ---
                   Variable  ADF Statistic   p-value  Stationary
0  orig_appreciation_return      -2.653460  0.082444       False
1  orig_available_df_direct      -2.252702  0.187676       False
2            orig_demand_sf      -1.888699  0.337396       False
3     orig_net_delivered_sf      -2.573779  0.0

/Users/elyas/Applications/anaconda3/envs/gatech_02/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)


ValueError: maxlags is too large for the number of observations and the number of equations. The largest model cannot be estimated.